In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing import image
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import pickle

# Paths
input_folder = r"C:\Users\gopal\Desktop\both"
svm_model_path = "svm_model_with_rgb.pkl"

# Load pre-trained MobileNetV2 for feature extraction
base_model = MobileNetV2(weights="imagenet", include_top=False, pooling="avg")

# Function to extract features from an image
def extract_features(img_path):
    # Load image for MobileNetV2 feature extraction
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = tf.keras.applications.mobilenet_v2.preprocess_input(img_array)
    features = base_model.predict(img_array)
    
    # Flatten MobileNetV2 features
    mobilenet_features = features.flatten()
    
    # Extract RGB features: mean of each channel
    img = np.array(img)  # Load the image again for RGB analysis
    r_mean = np.mean(img[:, :, 0])  # Mean of the red channel
    g_mean = np.mean(img[:, :, 1])  # Mean of the green channel
    b_mean = np.mean(img[:, :, 2])  # Mean of the blue channel
    
    # Combine MobileNetV2 features with RGB features
    combined_features = np.concatenate([mobilenet_features, [r_mean, g_mean, b_mean]])
    
    return combined_features

# Load dataset and extract features
image_paths = []
labels = []

# Assuming images are stored in "drowsy" and "non-drowsy" subfolders
for category in ["drowsy", "non-drowsy"]:
    category_path = os.path.join(input_folder, category)
    for img_name in os.listdir(category_path):
        img_path = os.path.join(category_path, img_name)
        image_paths.append(img_path)
        labels.append(0 if category == "drowsy" else 1)  # 0 = Drowsy, 1 = Non-Drowsy

# Extract features from all images
features = np.array([extract_features(img) for img in image_paths])
labels = np.array(labels)

# Split dataset into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, random_state=42)

# Train an SVM classifier
svm = SVC(kernel="linear", probability=True)
svm.fit(X_train, y_train)

# Save the trained model
with open(svm_model_path, "wb") as file:
    pickle.dump(svm, file)

print("SVM model trained and saved successfully!")

# Evaluate the model
y_pred = svm.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

# Print detailed classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Drowsy", "Non-Drowsy"]))

# Function to predict drowsiness for a new image
def predict_drowsiness(img_path):
    features = extract_features(img_path).reshape(1, -1)
    label = svm.predict(features)[0]
    return "Drowsy" if label == 0 else "Non-Drowsy"

# Example usage
image_path = r"C:\Users\gopal\Pictures\Camera Roll\WIN_20250226_16_06_38_Pro.jpg"
prediction = predict_drowsiness(image_path)
print(f"The image is classified as: {prediction}")
